In [ ]:
import graphite


In [ ]:
dataset = graphite.Dataset(source='washington',
                           level='line',
                           training_ratio=None,
                           validation_ratio=None,
                           test_ratio=None,
                           lazy_mode=False,
                           seed=42)
print(dataset)


In [ ]:
augmentor = graphite.Augmentor(#elastic_transform=[0.99, 43, 0.75],
                               erosion=[0.99, 5, 1],
                               dilation=[0.99, 2, 1],
                            #    mixup=[0.99, 0.3, 1],
                            #    perspective_transform=[0.99, 0.4],
                            #    salt_and_pepper=[0.99, 0.3],
                            #    gaussian_blur=[0.99, 11, 1],
                            #    shearing=[0.99, 30],
                               scaling=[0.99, 0.05],
                               rotation=[0.99, 0.5],
                               translation=[0.99, 0.05, 0.05],
                               seed=42)
print(augmentor)


In [ ]:
model = graphite.Model(network='flor', tokenizer=dataset.tokenizer, seed=42)
print(model)

model.compile(learning_rate=1e-3, run_index=None)


In [ ]:
training_data, training_steps = dataset.get_generator(partition='training', batch_size=16, augmentor=augmentor)
validation_data, validation_steps = dataset.get_generator(partition='validation', batch_size=16, augmentor=None)

history = model.fit(training_data=training_data,
                    training_steps=training_steps,
                    validation_data=validation_data,
                    validation_steps=validation_steps,
                    plateau_cooldown=0,
                    plateau_factor=0.1,
                    plateau_patience=15,
                    patience=20,
                    epochs=2,
                    verbose=1)


In [ ]:
test_data, test_steps = dataset.get_generator(partition='test', batch_size=16, augmentor=None)

predicts, probabilities = model.predict(test_data=test_data,
                                        test_steps=test_steps,
                                        top_paths=10,
                                        beam_width=15,
                                        ctc_decode=True,
                                        token_decode=True,
                                        verbose=1)

print(predicts)
print(predicts.shape, probabilities.shape)
print(predicts[0][0], probabilities[0][0])


In [ ]:
print(model.logger)
